In [1]:
from IPython.display import clear_output

In [2]:
# No need to run this on colab. These libraries come pre-installed on colab
# %pip install torch torchvision torchaudio

# Content:

In this demo, we will take do some AI-based code generation, like the kind done my github co-pilot or codenium or other code completion services.

The model we will use is codeLlama. CodeLlama models are basically llama-v2 models fine tuned for coding tasks. Code llama is available in different sizes and we'll use the 7B params model variant.


For this, we need to install the library and to download the model weights file. The file can be downloaded from huggingface [repo](https://huggingface.co/TheBloke/CodeLlama-7B-GGUF) of [TheBloke](https://huggingface.co/TheBloke). Credits to him for quantizing the model, saving it in different formats like GGML and GGUF and sharing with the community. He has a lot of other models on his channel that you can check out, including different versions of llama

## Downloading model file

In [8]:
# !wget https://huggingface.co/TheBloke/CodeLlama-7B-GGUF/resolve/main/codellama-7b.Q5_K_M.gguf
# 
# clear_output()

## Installing llama-cpp-python

installing supports different versions of hardware acceleration.

We will go with Cuda. Checkout the [Github Repo](https://github.com/abetlen/llama-cpp-python) for more options

In [9]:
# !CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python==0.2.74  # This takes a few mins when building wheel. Be patient.

## Running Llama-v2

In [10]:
import json

from llama_cpp import Llama

In [11]:
model = Llama(
    "codellama-7b.Q5_K_M.gguf",
    n_gpu_layers=-1, # To use GPU
    n_ctx=2048,
)

clear_output()

Let's try a code generation example. How about a function to open an RGB image, convert it to greyscale and then save it.

In [12]:
prompt = """
from PIL import Image

def convert_to_greyscale(input_path, output_path):
"""

In [13]:
output = model.__call__(
    prompt,
    max_tokens=None,  # sets no length limit
    temperature=0.5,
)


llama_print_timings:        load time =    1676.09 ms
llama_print_timings:      sample time =      20.39 ms /   147 runs   (    0.14 ms per token,  7210.83 tokens per second)
llama_print_timings: prompt eval time =    1676.00 ms /    29 tokens (   57.79 ms per token,    17.30 tokens per second)
llama_print_timings:        eval time =    1845.58 ms /   146 runs   (   12.64 ms per token,    79.11 tokens per second)
llama_print_timings:       total time =    3719.68 ms /   175 tokens


In [14]:
output.keys()

dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage'])

In [15]:
print(output["choices"][0]['text'])

    image = Image.open(input_path)
    width, height = image.size
    if width != height:
        raise ValueError("Image must be square")

    pixels = image.load()
    for x in range(width):
        for y in range(height):
            r, g, b = pixels[x, y]
            luminance = int((r + g + b) / 3)
            pixels[x, y] = (luminance, luminance, luminance)
    image.save(output_path)

if __name__ == "__main__":
    convert_to


Let's try a translation example now. Something a little more complex.

How about a deep learning example

In [16]:
prompt = """
# Simple script to download MNIST, Create a pytorch classifier class, and training it for the MNIST.

import torch
"""

In [17]:
output = model(prompt, max_tokens=None, temperature=0.1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1676.09 ms
llama_print_timings:      sample time =     295.57 ms /  2012 runs   (    0.15 ms per token,  6807.16 tokens per second)
llama_print_timings: prompt eval time =     317.45 ms /    33 tokens (    9.62 ms per token,   103.95 tokens per second)
llama_print_timings:        eval time =   27407.16 ms /  2011 runs   (   13.63 ms per token,    73.37 tokens per second)
llama_print_timings:       total time =   31863.55 ms /  2044 tokens


In [18]:
print(output["choices"][0]['text'])

from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Download the dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

# Create the dataloaders
train_loader = DataLoader(train_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=1000)

# Define a simple classifier
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, 10)
    
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Create the model and optimizer
model = Classifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Train the model
for epoch in range